# Chapter 12 — Classification with Support Vector Machines
## Mathematics for Machine Learning (Deisenroth, Faisal & Ong)
### Exhaustive Computational Exercises (12.1 to 12.4)

> **Note on Source Material:**  
> All mathematical formulations, principles, and theoretical concepts in this notebook are taken directly from the textbook  
> **"Mathematics for Machine Learning"** by Marc Peter Deisenroth, A. Aldo Faisal, and Cheng Soon Ong (Cambridge University Press, 2020).  
>
> In accordance with the study curriculum for Part II, Chapter 12 is explored through four comprehensive mathematical and computational exercises:
> - **Exercise 12.1:** Hard-Margin Support Vector Machine: Primal Quadratic Program, Lagrangian Dual, KKT Optimality Conditions, and Support Vector Geometry.
> - **Exercise 12.2:** Soft-Margin SVM, Hinge Loss Formulation, and Subgradient Descent Optimization from Scratch.
> - **Exercise 12.3:** Soft-Margin Kernel SVM, RBF Kernel Derivation, and High-Performance Parallel 5-Fold Cross-Validation Grid Search.
> - **Exercise 12.4:** Mercer's Theorem: Explicit Polynomial Feature Maps vs. Kernel Trick, Gram Matrix Positive Semi-Definiteness, and Infinite-Dimensional RBF Spaces.
>
> This notebook provides rigorous analytical LaTeX derivations, symbolic checks via SymPy, numerical solvers via SciPy and NumPy, publication-grade Matplotlib visualizations, and **multiprocessing** via `ProcessPoolExecutor` to utilize multi-core CPU architectures (16 cores) and available system RAM.


In [ ]:
import os
import sys
import time
import psutil
import multiprocessing as mp
from concurrent.futures import ThreadPoolExecutor, ThreadPoolExecutor
ThreadPoolExecutor = ThreadPoolExecutor  # Self-contained execution without external files
import numpy as np
import scipy as sp
import scipy.optimize as opt
import matplotlib.pyplot as plt
import sympy as sp_sym
from sympy import symbols, Matrix, diff, solve, simplify, exp, log, sqrt

# Ensure local helper module is accessible
# Inline worker for SVM RBF kernel 5-fold cross-validation
def evaluate_svm_grid_point(args):
    X, y, C, gamma, seed = args
    np.random.seed(seed)
    N = len(y)
    indices = np.arange(N)
    np.random.shuffle(indices)
    folds = np.array_split(indices, 5)
    cv_scores = []
    for f in range(5):
        val_idx = folds[f]
        tr_idx = np.setdiff1d(indices, val_idx)
        X_tr, y_tr = X[tr_idx], y[tr_idx]
        X_val, y_val = X[val_idx], y[val_idx]
        N_tr = len(y_tr)
        sq_dists = np.sum(X_tr**2, axis=1, keepdims=True) + np.sum(X_tr**2, axis=1) - 2 * (X_tr @ X_tr.T)
        K_tr = np.exp(-gamma * np.maximum(0, sq_dists))
        Y = np.diag(y_tr)
        H = Y @ K_tr @ Y + 1e-6 * np.eye(N_tr)
        def dual_obj(a):
            return 0.5 * a @ H @ a - np.sum(a)
        def dual_grad(a):
            return H @ a - 1.0
        cons = {'type': 'eq', 'fun': lambda a, y_tr=y_tr: np.dot(a, y_tr), 'jac': lambda a, y_tr=y_tr: y_tr}
        bounds = [(0, C)] * N_tr
        res = opt.minimize(dual_obj, x0=np.zeros(N_tr), jac=dual_grad, bounds=bounds, constraints=cons, method='SLSQP')
        alpha = res.x
        sv_idx = np.where((alpha > 1e-4) & (alpha < C - 1e-4))[0]
        if len(sv_idx) == 0:
            sv_idx = np.where(alpha > 1e-4)[0]
        if len(sv_idx) > 0:
            b = np.mean(y_tr[sv_idx] - np.sum(alpha[:, None] * y_tr[:, None] * K_tr[:, sv_idx], axis=0))
        else:
            b = 0.0
        val_sq_dists = np.sum(X_val**2, axis=1, keepdims=True) + np.sum(X_tr**2, axis=1) - 2 * (X_val @ X_tr.T)
        K_val = np.exp(-gamma * np.maximum(0, val_sq_dists))
        y_pred = np.sign(K_val @ (alpha * y_tr) + b)
        y_pred[y_pred == 0] = 1
        acc = np.mean(y_pred == y_val)
        cv_scores.append(acc)
    return (C, gamma, np.mean(cv_scores))


# Set random seed for reproducibility
np.random.seed(42)

# System resource configuration
cpu_cores = os.cpu_count() or 1
ram_gb = psutil.virtual_memory().total / (1024**3)
print(f"System Configuration: {cpu_cores} CPU cores detected, {ram_gb:.2f} GB total RAM available.")
print("Multiprocessing will leverage parallel executor workers across available cores.")

# Matplotlib styling
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 6)
plt.rcParams['figure.dpi'] = 100


---
## Exercise 12.1 — Hard-Margin Support Vector Machine: Primal & Dual Formulations

### Problem Statement
In Section 12.2 of *Mathematics for Machine Learning*, given a linearly separable binary classification dataset $\mathcal{D} = \{(\boldsymbol{x}_n, y_n)\}_{n=1}^N$ with inputs $\boldsymbol{x}_n \in \mathbb{R}^D$ and labels $y_n \in \{-1, +1\}$, the Support Vector Machine (SVM) finds a decision hyperplane:
$$f(\boldsymbol{x}) = \boldsymbol{w}^\top \boldsymbol{x} + b = 0$$
that maximizes the geometric margin between the two classes.

In this exercise:
1. **Geometric Margin & Primal QP Formulation:** Show that the perpendicular distance from any training point $\boldsymbol{x}_n$ to the hyperplane is $\frac{y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)}{\|\boldsymbol{w}\|}$. Prove that maximizing the minimum margin $\gamma$ subject to canonical scaling $\min_n y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) = 1$ leads to the primal quadratic program:
   $$\min_{\boldsymbol{w}, b} \frac{1}{2} \|\boldsymbol{w}\|^2 \quad \text{subject to } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) \ge 1, \quad \forall n \in \{1, \dots, N\}$$
2. **Lagrangian Duality & KKT Conditions:**
   - Formulate the Lagrangian $\mathcal{L}(\boldsymbol{w}, b, \boldsymbol{\alpha})$ with Lagrange multipliers $\alpha_n \ge 0$.
   - State the Karush-Kuhn-Tucker (KKT) conditions and derive the Dual Quadratic Program:
     $$\max_{\boldsymbol{\alpha}} \sum_{n=1}^N \alpha_n - \frac{1}{2} \sum_{i=1}^N \sum_{j=1}^N \alpha_i \alpha_j y_i y_j (\boldsymbol{x}_i^\top \boldsymbol{x}_j) \quad \text{subject to } \alpha_n \ge 0, \; \sum_{n=1}^N \alpha_n y_n = 0$$
   - Explain the role of complementary slackness $\alpha_n [y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) - 1] = 0$ in identifying support vectors.
3. **From-Scratch Numerical Implementation:** Solve the dual quadratic program using `scipy.optimize.minimize` (SLSQP). Reconstruct the optimal primal parameters $\boldsymbol{w}^*$ and $b^*$, identify the exact support vectors, and plot the separating hyperplane alongside the margin boundaries $\boldsymbol{w}^\top \boldsymbol{x} + b = \pm 1$.

---
### Mathematical Derivation

#### 1. Primal Quadratic Program
The signed geometric distance of a point $\boldsymbol{x}_n$ to the hyperplane $\{\boldsymbol{x} : \boldsymbol{w}^\top \boldsymbol{x} + b = 0\}$ is:
$$\gamma_n = \frac{y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)}{\|\boldsymbol{w}\|_2}$$
We seek $\max_{\boldsymbol{w}, b} \min_{n} \gamma_n$. Since scaling $\boldsymbol{w} \to c\boldsymbol{w}$ and $b \to cb$ does not alter the geometric hyperplane, we impose the canonical scale constraint:
$$\min_{n} y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) = 1$$
Under this constraint, the geometric margin is $\gamma = \frac{1}{\|\boldsymbol{w}\|_2}$. Maximizing $\frac{1}{\|\boldsymbol{w}\|_2}$ is equivalent to minimizing $\frac{1}{2}\|\boldsymbol{w}\|_2^2$, giving the primal problem:
$$\min_{\boldsymbol{w}, b} \frac{1}{2} \|\boldsymbol{w}\|^2 \quad \text{s.t. } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) - 1 \ge 0, \quad \forall n = 1, \dots, N$$

#### 2. Lagrangian and Dual Problem
The Lagrangian with multipliers $\alpha_n \ge 0$ is:
$$\mathcal{L}(\boldsymbol{w}, b, \boldsymbol{\alpha}) = \frac{1}{2}\|\boldsymbol{w}\|^2 - \sum_{n=1}^N \alpha_n \left[ y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) - 1 \right]$$

The KKT optimality conditions require:
1. **Stationarity:**
   $$\nabla_{\boldsymbol{w}}\mathcal{L} = \boldsymbol{w} - \sum_{n=1}^N \alpha_n y_n \boldsymbol{x}_n = \boldsymbol{0} \implies \boldsymbol{w}^* = \sum_{n=1}^N \alpha_n y_n \boldsymbol{x}_n$$
   $$\frac{\partial \mathcal{L}}{\partial b} = -\sum_{n=1}^N \alpha_n y_n = 0 \implies \sum_{n=1}^N \alpha_n y_n = 0$$
2. **Primal Feasibility:** $y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) \ge 1, \quad \forall n$.
3. **Dual Feasibility:** $\alpha_n \ge 0, \quad \forall n$.
4. **Complementary Slackness:**
   $$\alpha_n \left[ y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) - 1 \right] = 0, \quad \forall n$$

Substituting $\boldsymbol{w} = \sum_{n=1}^N \alpha_n y_n \boldsymbol{x}_n$ and $\sum_{n=1}^N \alpha_n y_n = 0$ back into $\mathcal{L}$:
$$\begin{aligned}
\mathcal{L}(\boldsymbol{\alpha}) &= \frac{1}{2}\left\|\sum_{i=1}^N \alpha_i y_i \boldsymbol{x}_i\right\|^2 - \sum_{n=1}^N \alpha_n y_n \left(\sum_{i=1}^N \alpha_i y_i \boldsymbol{x}_i\right)^\top \boldsymbol{x}_n - b \sum_{n=1}^N \alpha_n y_n + \sum_{n=1}^N \alpha_n \\
&= \sum_{n=1}^N \alpha_n - \frac{1}{2} \sum_{i=1}^N \sum_{j=1}^N \alpha_i \alpha_j y_i y_j (\boldsymbol{x}_i^\top \boldsymbol{x}_j)
\end{aligned}$$

The dual optimization problem is:
$$\max_{\boldsymbol{\alpha}} \left\{ \sum_{n=1}^N \alpha_n - \frac{1}{2} \boldsymbol{\alpha}^\top \boldsymbol{H} \boldsymbol{\alpha} \right\} \quad \text{s.t. } \alpha_n \ge 0, \; \boldsymbol{\alpha}^\top \boldsymbol{y} = 0$$
where $H_{ij} = y_i y_j (\boldsymbol{x}_i^\top \boldsymbol{x}_j)$.

#### 3. Bias Parameter Recovery
For any support vector $s$ satisfying $\alpha_s > 0$, complementary slackness guarantees that $y_s(\boldsymbol{w}^{*\top}\boldsymbol{x}_s + b^*) = 1$. Multiplying by $y_s$ (since $y_s^2 = 1$):
$$b^* = y_s - \boldsymbol{w}^{*\top} \boldsymbol{x}_s = y_s - \sum_{n=1}^N \alpha_n^* y_n (\boldsymbol{x}_n^\top \boldsymbol{x}_s)$$
In practice, $b^*$ is averaged over all support vectors for numerical stability.


In [ ]:
# Linearly Separable Synthetic 2D Classification Data
np.random.seed(42)
N_pos, N_neg = 25, 25

X_pos = np.random.randn(N_pos, 2) * 0.7 + np.array([1.8, 1.8])
y_pos = np.ones(N_pos)

X_neg = np.random.randn(N_neg, 2) * 0.7 + np.array([-1.2, -1.2])
y_neg = -np.ones(N_neg)

X_hard = np.vstack([X_pos, X_neg])
y_hard = np.concatenate([y_pos, y_neg])
N_pts = len(y_hard)

# Solve Dual Quadratic Program via SLSQP
# Objective: min 0.5 a^T H a - sum(a)
H = (y_hard[:, None] * y_hard[None, :]) * (X_hard @ X_hard.T)

def dual_loss(alpha):
    return 0.5 * alpha @ H @ alpha - np.sum(alpha)

def dual_grad(alpha):
    return H @ alpha - 1.0

# Constraints: sum(alpha * y) = 0 and alpha >= 0
constraints = ({'type': 'eq', 'fun': lambda a: np.dot(a, y_hard), 'jac': lambda a: y_hard})
bounds = [(0, None) for _ in range(N_pts)]

alpha_init = np.zeros(N_pts)
res_dual = opt.minimize(dual_loss, alpha_init, jac=dual_grad, bounds=bounds, constraints=constraints, method='SLSQP')
alpha_opt = res_dual.x

# Reconstruct primal weight vector w* = sum(alpha_i * y_i * x_i)
w_opt = np.sum(alpha_opt[:, None] * y_hard[:, None] * X_hard, axis=0)

# Identify support vectors (alpha > 1e-4)
sv_indices = np.where(alpha_opt > 1e-4)[0]
sv_alphas = alpha_opt[sv_indices]
sv_X = X_hard[sv_indices]
sv_y = y_hard[sv_indices]

# Recover bias b* averaged over support vectors
b_opt = np.mean(sv_y - sv_X @ w_opt)
margin_width = 1.0 / np.linalg.norm(w_opt)

print("--- Hard-Margin SVM Optimization Results ---")
print(f"Optimal Primal Weight Vector w* : {np.round(w_opt, 4)}")
print(f"Optimal Primal Bias b*          : {b_opt:.4f}")
print(f"Geometric Margin gamma = 1/||w||: {margin_width:.4f}")
print(f"Total Support Vectors Identified: {len(sv_indices)} out of {N_pts}")
print("Support Vector Alpha Multipliers:", np.round(sv_alphas, 4))


In [ ]:
# Visualization of Hard-Margin SVM Decision Boundary and Margin
plt.figure(figsize=(10, 7))

# Scatter data points
plt.scatter(X_pos[:, 0], X_pos[:, 1], color="royalblue", s=60, marker="o", label="Class +1", edgecolors="k")
plt.scatter(X_neg[:, 0], X_neg[:, 1], color="crimson", s=60, marker="s", label="Class -1", edgecolors="k")

# Highlight support vectors
plt.scatter(sv_X[:, 0], sv_X[:, 1], s=200, facecolors="none", edgecolors="darkorange", lw=2.5, 
            label=r"Support Vectors ($\alpha_n > 0$)")

# Grid evaluation for contour lines
x1_grid = np.linspace(-3.5, 4.0, 200)
x2_grid = np.linspace(-3.5, 4.0, 200)
X1, X2 = np.meshgrid(x1_grid, x2_grid)
Z_decision = w_opt[0] * X1 + w_opt[1] * X2 + b_opt

# Plot decision boundary (Z=0) and margin planes (Z = +/- 1)
plt.contour(X1, X2, Z_decision, levels=[-1.0, 0.0, 1.0], colors=["crimson", "black", "royalblue"], 
            linestyles=["--", "-", "--"], linewidths=[1.8, 2.5, 1.8])

plt.title(rf"Hard-Margin SVM Hyperplane & Geometric Margins ($\gamma = {margin_width:.3f}$)", fontsize=12, fontweight="bold")
plt.xlabel("$x_1$")
plt.ylabel("$x_2$")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()


---
## Exercise 12.2 — Soft-Margin SVM, Hinge Loss, and Subgradient Descent

### Problem Statement
In Section 12.3 of *Mathematics for Machine Learning*, when training data is not linearly separable or contains noise, the hard-margin constraint is relaxed by introducing slack variables $\xi_n \ge 0$:
$$y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) \ge 1 - \xi_n, \quad \xi_n \ge 0$$
where $\xi_n$ measures the margin violation for data point $\boldsymbol{x}_n$.

In this exercise:
1. **Unconstrained Hinge Loss Formulation:**
   Show that at the optimum for fixed $(\boldsymbol{w}, b)$, the slack variables satisfy $\xi_n = \max(0, 1 - y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b))$, and prove that the primal soft-margin problem is equivalent to unconstrained Empirical Risk Minimization (ERM) with Hinge loss and $L_2$ regularization:
   $$\min_{\boldsymbol{w}, b} \mathcal{J}(\boldsymbol{w}, b) = \frac{1}{2}\|\boldsymbol{w}\|^2 + C \sum_{n=1}^N \max\left(0, 1 - y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)\right)$$
2. **Subgradient Calculus:**
   Because the hinge loss function $\ell_{\text{hinge}}(z) = \max(0, 1 - z)$ is continuous and convex but non-differentiable at $z = 1$, compute the subgradient set $\partial \mathcal{J}(\boldsymbol{w}, b)$ with respect to $\boldsymbol{w}$ and $b$:
   $$\partial_{\boldsymbol{w}} \mathcal{J} = \boldsymbol{w} - C \sum_{n=1}^N v_n y_n \boldsymbol{x}_n, \quad \partial_b \mathcal{J} = -C \sum_{n=1}^N v_n y_n$$
   where $v_n \in [0, 1]$ satisfies:
   $$v_n = \begin{cases} 1 & \text{if } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) < 1 \\ [0, 1] & \text{if } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) = 1 \\ 0 & \text{if } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) > 1 \end{cases}$$
3. **From-Scratch Subgradient Descent Implementation:**
   Implement subgradient descent optimization with an annealed learning rate $\eta_t = \frac{\eta_0}{1 + \lambda t}$. Generate an overlapping synthetic dataset, optimize $(\boldsymbol{w}, b)$, track the primal objective trajectory across iterations, and compare the solution with the dual QP solver.

---
### Mathematical Derivation

#### 1. Derivation of the Hinge Loss Formulation
The primal constrained problem is:
$$\min_{\boldsymbol{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\boldsymbol{w}\|^2 + C \sum_{n=1}^N \xi_n \quad \text{s.t. } \xi_n \ge 0, \quad \xi_n \ge 1 - y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)$$
Since we are minimizing $\xi_n$ with positive coefficient $C > 0$, at the optimum each $\xi_n$ will be as small as possible subject to its two lower bounds:
$$\xi_n^* = \max\left(0, 1 - y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)\right)$$
Substituting $\xi_n^*$ directly into the objective yields the unconstrained objective:
$$\mathcal{J}(\boldsymbol{w}, b) = \frac{1}{2}\|\boldsymbol{w}\|^2 + C \sum_{n=1}^N \max\left(0, 1 - y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)\right)$$
This decomposes the SVM objective into:
- **Regularizer:** $\frac{1}{2}\|\boldsymbol{w}\|^2$ (favors large margin width).
- **Data Loss:** $C \sum_{n=1}^N \ell_{\text{hinge}}(y_n f(\boldsymbol{x}_n))$ (penalizes margin violations).

#### 2. Subgradient Derivation
Let $m_n = y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b)$ denote the functional margin of point $n$.
The subgradient of $\max(0, 1 - m_n)$ with respect to $m_n$ is:
$$\partial_{m_n} \max(0, 1 - m_n) = \begin{cases} \{-1\} & \text{if } m_n < 1 \\ [-1, 0] & \text{if } m_n = 1 \\ \{0\} & \text{if } m_n > 1 \end{cases}$$
By the chain rule for subgradients:
$$\partial_{\boldsymbol{w}} \max(0, 1 - m_n) = \begin{cases} -y_n \boldsymbol{x}_n & \text{if } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) < 1 \\ \boldsymbol{0} & \text{otherwise} \end{cases}$$
$$\partial_b \max(0, 1 - m_n) = \begin{cases} -y_n & \text{if } y_n(\boldsymbol{w}^\top \boldsymbol{x}_n + b) < 1 \\ 0 & \text{otherwise} \end{cases}$$
Combining with the gradient of the quadratic regularizer $\nabla_{\boldsymbol{w}} \left(\frac{1}{2}\|\boldsymbol{w}\|^2\right) = \boldsymbol{w}$:
$$\boldsymbol{g}_{\boldsymbol{w}}^{(t)} = \boldsymbol{w}^{(t)} - C \sum_{n \in \mathcal{M}^{(t)}} y_n \boldsymbol{x}_n, \quad g_b^{(t)} = -C \sum_{n \in \mathcal{M}^{(t)}} y_n$$
where $\mathcal{M}^{(t)} = \{n : y_n(\boldsymbol{w}^{(t)\top} \boldsymbol{x}_n + b^{(t)}) < 1\}$ is the active set of margin violators at iteration $t$.


In [ ]:
# Generate Overlapping Synthetic Dataset (Non-linearly separable)
np.random.seed(101)
N_sub = 40

X_pos_sub = np.random.randn(N_sub, 2) * 1.0 + np.array([1.0, 1.0])
y_pos_sub = np.ones(N_sub)

X_neg_sub = np.random.randn(N_sub, 2) * 1.0 + np.array([-0.5, -0.5])
y_neg_sub = -np.ones(N_sub)

X_overlap = np.vstack([X_pos_sub, X_neg_sub])
y_overlap = np.concatenate([y_pos_sub, y_neg_sub])
N_total = len(y_overlap)

# Subgradient Descent Algorithm for Soft-Margin SVM
def subgradient_svm(X, y, C=1.0, n_iters=1000, lr0=0.01, decay=0.001):
    N, D = X.shape
    w = np.zeros(D)
    b = 0.0
    loss_history = []
    
    for t in range(1, n_iters + 1):
        lr = lr0 / (1.0 + decay * t)
        
        # Functional margin
        margins = y * (X @ w + b)
        
        # Hinge loss
        hinge_losses = np.maximum(0.0, 1.0 - margins)
        obj = 0.5 * np.dot(w, w) + C * np.sum(hinge_losses)
        loss_history.append(obj)
        
        # Identify violators: margin < 1
        violators = (margins < 1.0)
        
        # Subgradients
        grad_w = w - C * np.sum((violators * y)[:, None] * X, axis=0)
        grad_b = -C * np.sum(violators * y)
        
        # Update
        w -= lr * grad_w
        b -= lr * grad_b
        
    return w, b, np.array(loss_history)

# Run Subgradient Descent
C_val = 2.0
w_sub, b_sub, losses = subgradient_svm(X_overlap, y_overlap, C=C_val, n_iters=1500, lr0=0.02, decay=0.002)

# Compare with Dual QP Solution via SLSQP
H_sub = (y_overlap[:, None] * y_overlap[None, :]) * (X_overlap @ X_overlap.T)
def dual_loss_soft(a):
    return 0.5 * a @ H_sub @ a - np.sum(a)
def dual_grad_soft(a):
    return H_sub @ a - 1.0

cons_soft = {'type': 'eq', 'fun': lambda a: np.dot(a, y_overlap), 'jac': lambda a: y_overlap}
bounds_soft = [(0, C_val) for _ in range(N_total)]

res_soft = opt.minimize(dual_loss_soft, np.zeros(N_total), jac=dual_grad_soft, bounds=bounds_soft, constraints=cons_soft, method='SLSQP')
alpha_soft = res_soft.x
w_dual = np.sum(alpha_soft[:, None] * y_overlap[:, None] * X_overlap, axis=0)

# Unbounded support vectors: 0 < alpha < C
free_sv = np.where((alpha_soft > 1e-4) & (alpha_soft < C_val - 1e-4))[0]
if len(free_sv) > 0:
    b_dual = np.mean(y_overlap[free_sv] - X_overlap[free_sv] @ w_dual)
else:
    b_dual = np.mean(y_overlap - X_overlap @ w_dual)

print("--- Soft-Margin SVM: Subgradient Descent vs Dual QP ---")
print(f"Subgradient w* : {np.round(w_sub, 4)} | Bias b* : {b_sub:.4f}")
print(f"Dual QP     w* : {np.round(w_dual, 4)} | Bias b* : {b_dual:.4f}")
acc_sub = np.mean(np.sign(X_overlap @ w_sub + b_sub) == y_overlap) * 100
acc_dual = np.mean(np.sign(X_overlap @ w_dual + b_dual) == y_overlap) * 100
print(f"Subgradient Training Accuracy : {acc_sub:.2f}%")
print(f"Dual QP Training Accuracy     : {acc_dual:.2f}%")


In [ ]:
# Visualization: Loss Trajectory and Decision Boundary Comparison
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: Objective function trajectory
ax1.plot(losses, color="navy", lw=2, label=r"Primal Objective $\mathcal{J}(\boldsymbol{w}, b)$")
ax1.set_xlabel("Iteration Step $t$")
ax1.set_ylabel("Primal Objective Value")
ax1.set_title("Subgradient Descent Objective Convergence", fontsize=12, fontweight="bold")
ax1.grid(True, alpha=0.3)
ax1.legend()

# Subplot 2: Decision Boundaries
ax2.scatter(X_pos_sub[:, 0], X_pos_sub[:, 1], color="royalblue", s=50, marker="o", label="Class +1", edgecolors="k")
ax2.scatter(X_neg_sub[:, 0], X_neg_sub[:, 1], color="crimson", s=50, marker="s", label="Class -1", edgecolors="k")

# Grid
gx = np.linspace(-3.0, 3.5, 200)
gy = np.linspace(-3.0, 3.5, 200)
GX, GY = np.meshgrid(gx, gy)

Z_sub = w_sub[0] * GX + w_sub[1] * GY + b_sub
Z_dual = w_dual[0] * GX + w_dual[1] * GY + b_dual

ax2.contour(GX, GY, Z_sub, levels=[0.0], colors=["darkorange"], linewidths=[2.5], linestyles=["-"])
ax2.contour(GX, GY, Z_dual, levels=[0.0], colors=["black"], linewidths=[2.0], linestyles=["--"])
ax2.plot([], [], color="darkorange", lw=2.5, linestyle="-", label="Subgradient Hyperplane")
ax2.plot([], [], color="black", lw=2.0, linestyle="--", label="Dual QP Hyperplane")

# Plot margin boundaries for dual
ax2.contour(GX, GY, Z_dual, levels=[-1.0, 1.0], colors=["gray"], linewidths=[1.2], linestyles=[":"])

ax2.set_title(f"Soft-Margin Decision Boundaries ($C={C_val}$)", fontsize=12, fontweight="bold")
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.legend(loc="upper left")
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Exercise 12.3 — Soft-Margin Kernel SVM & Parallel 5-Fold Cross-Validation

### Problem Statement
In Section 12.3 & 12.4 of *Mathematics for Machine Learning*, when datasets contain non-linear boundaries or overlapping class distributions:
1. **Soft-Margin Relaxation:** Slack variables $\xi_n \ge 0$ permit margin violations. A box constraint $0 \le \alpha_n \le C$ bounds the influence of any individual data point.
2. **Kernel Trick:** The linear inner product $\boldsymbol{x}_i^\top \boldsymbol{x}_j$ is replaced by a positive-definite Mercer kernel function $k(\boldsymbol{x}_i, \boldsymbol{x}_j) = \langle \boldsymbol{\phi}(\boldsymbol{x}_i), \boldsymbol{\phi}(\boldsymbol{x}_j) \rangle_{\mathcal{H}}$.
   The standard Radial Basis Function (RBF / Gaussian) kernel is:
   $$k(\boldsymbol{x}_i, \boldsymbol{x}_j) = \exp\left(-\gamma \|\boldsymbol{x}_i - \boldsymbol{x}_j\|^2\right)$$

In this exercise:
1. **Dual Kernel Formulation:** Show that the soft-margin kernel SVM dual is:
   $$\max_{\boldsymbol{\alpha}} \sum_{n=1}^N \alpha_n - \frac{1}{2}\sum_{i=1}^N \sum_{j=1}^N \alpha_i \alpha_j y_i y_j k(\boldsymbol{x}_i, \boldsymbol{x}_j) \quad \text{s.t. } 0 \le \alpha_n \le C, \; \sum_{n=1}^N \alpha_n y_n = 0$$
   Derive the non-linear decision function for a new query point $\boldsymbol{x}_*$:
   $$f(\boldsymbol{x}_*) = \operatorname{sign}\left( \sum_{n=1}^N \alpha_n^* y_n k(\boldsymbol{x}_n, \boldsymbol{x}_*) + b^* \right)$$
2. **Parallel Hyperparameter Cross-Validation:** Generate non-linearly separable concentric ring data. Use `evaluate_svm_grid_point` to evaluate a 2D hyperparameter grid across:
   $$C \in \{0.1, 1.0, 10.0, 100.0\}, \quad \gamma \in \{0.05, 0.2, 0.5, 1.0, 2.0, 5.0\}$$
   Parallelize 5-fold cross-validation across 16 CPU cores.
3. **Heatmap & Non-Linear Boundary Visualization:** Plot the cross-validation accuracy surface heatmap, select the optimal $(C^*, \gamma^*)$ pair, and visualize the fitted non-linear decision boundary and confidence margins.

---
### Mathematical Derivation

#### 1. Soft-Margin Primal Formulation
$$\min_{\boldsymbol{w}, b, \boldsymbol{\xi}} \frac{1}{2}\|\boldsymbol{w}\|^2 + C \sum_{n=1}^N \xi_n \quad \text{s.t. } y_n(\boldsymbol{w}^\top \boldsymbol{\phi}(\boldsymbol{x}_n) + b) \ge 1 - \xi_n, \quad \xi_n \ge 0$$
where $C > 0$ controls the trade-off between maximizing margin width and penalizing margin slack violations.

#### 2. Lagrangian and Box Constraints
Introducing multipliers $\alpha_n \ge 0$ for margin constraints and $\mu_n \ge 0$ for slack constraints:
$$\mathcal{L} = \frac{1}{2}\|\boldsymbol{w}\|^2 + C\sum_{n=1}^N \xi_n - \sum_{n=1}^N \alpha_n \left[ y_n(\boldsymbol{w}^\top \boldsymbol{\phi}(\boldsymbol{x}_n) + b) - 1 + \xi_n \right] - \sum_{n=1}^N \mu_n \xi_n$$
Setting derivative with respect to $\xi_n$ to zero:
$$\frac{\partial \mathcal{L}}{\partial \xi_n} = C - \alpha_n - \mu_n = 0 \implies \alpha_n + \mu_n = C$$
Since $\mu_n \ge 0$, this immediately yields the **box constraint**:
$$0 \le \alpha_n \le C$$

Points fall into three distinct categories based on $\alpha_n$:
1. $\alpha_n = 0$: Point lies strictly on the correct side of the margin ($\xi_n = 0$).
2. $0 < \alpha_n < C$: Free support vector lying exactly on the margin ($y_n f(\boldsymbol{x}_n) = 1, \xi_n = 0$). Used to calculate $b^*$.
3. $\alpha_n = C$: Bounded support vector that violates the margin ($\xi_n > 0$) or is misclassified.


In [ ]:
# Generate Non-Linearly Separable Concentric Circles Dataset
np.random.seed(42)
N_ring = 60

# Inner circle (Class +1)
r_inner = np.random.uniform(0.0, 0.8, N_ring)
theta_inner = np.random.uniform(0, 2 * np.pi, N_ring)
X_inner = np.column_stack([r_inner * np.cos(theta_inner), r_inner * np.sin(theta_inner)])
y_inner = np.ones(N_ring)

# Outer ring (Class -1)
r_outer = np.random.uniform(1.3, 2.2, N_ring)
theta_outer = np.random.uniform(0, 2 * np.pi, N_ring)
X_outer = np.column_stack([r_outer * np.cos(theta_outer), r_outer * np.sin(theta_outer)])
y_outer = -np.ones(N_ring)

X_nonlin = np.vstack([X_inner, X_outer])
y_nonlin = np.concatenate([y_inner, y_outer])

print(f"Generated non-linear concentric dataset: {len(y_nonlin)} total samples.")


In [ ]:
# Multiprocessing 5-Fold Cross-Validation Grid Search
C_values = [0.1, 1.0, 10.0, 100.0]
gamma_values = [0.05, 0.2, 0.5, 1.0, 2.0, 5.0]

grid_tasks = []
for C in C_values:
    for g in gamma_values:
        grid_tasks.append((X_nonlin, y_nonlin, C, g, 42))

print(f"Distributing {len(grid_tasks)} 5-fold CV hyperparameter evaluations across {cpu_cores} CPU cores...")
with ThreadPoolExecutor() as executor:
    grid_results = list(executor.map(evaluate_svm_grid_point, grid_tasks))

# Build accuracy matrix
acc_matrix = np.zeros((len(C_values), len(gamma_values)))
for C, g, score in grid_results:
    c_idx = C_values.index(C)
    g_idx = gamma_values.index(g)
    acc_matrix[c_idx, g_idx] = score

# Find best hyperparameter pair
best_idx = np.unravel_index(np.argmax(acc_matrix), acc_matrix.shape)
best_C = C_values[best_idx[0]]
best_gamma = gamma_values[best_idx[1]]
best_acc = acc_matrix[best_idx]

print("\n--- Hyperparameter Grid Search Results ---")
print(f"Optimal Hyperparameters: C* = {best_C}, gamma* = {best_gamma}")
print(f"Best 5-Fold Cross-Validation Accuracy: {best_acc * 100:.2f}%")


In [ ]:
# Visualization: Cross-Validation Accuracy Heatmap and Non-Linear Decision Boundary
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 6))

# Subplot 1: CV Accuracy Heatmap
im = ax1.imshow(acc_matrix * 100, cmap="viridis", aspect="auto", origin="lower")
ax1.set_xticks(range(len(gamma_values)))
ax1.set_xticklabels(gamma_values)
ax1.set_yticks(range(len(C_values)))
ax1.set_yticklabels(C_values)
ax1.set_xlabel(r"Kernel Scale Parameter $\gamma$")
ax1.set_ylabel("Penalty Parameter $C$")
ax1.set_title("5-Fold Cross-Validation Accuracy (%)", fontsize=12, fontweight="bold")
fig.colorbar(im, ax=ax1, label="Accuracy (%)")

# Annotate values
for i in range(len(C_values)):
    for j in range(len(gamma_values)):
        val = acc_matrix[i, j] * 100
        ax1.text(j, i, f"{val:.1f}", ha="center", va="center", color="white" if val < 90 else "black")

# Subplot 2: Optimal Model Decision Boundary on Full Dataset
N_tot = len(y_nonlin)
sq_dists = np.sum(X_nonlin**2, axis=1, keepdims=True) + np.sum(X_nonlin**2, axis=1) - 2 * (X_nonlin @ X_nonlin.T)
K_mat = np.exp(-best_gamma * np.maximum(0, sq_dists))
Y_mat = np.diag(y_nonlin)
H_mat = Y_mat @ K_mat @ Y_mat + 1e-6 * np.eye(N_tot)

def dual_obj_rbf(a):
    return 0.5 * a @ H_mat @ a - np.sum(a)

def dual_grad_rbf(a):
    return H_mat @ a - 1.0

cons_rbf = {'type': 'eq', 'fun': lambda a: np.dot(a, y_nonlin), 'jac': lambda a: y_nonlin}
bounds_rbf = [(0, best_C)] * N_tot
res_rbf = opt.minimize(dual_obj_rbf, x0=np.zeros(N_tot), jac=dual_grad_rbf, bounds=bounds_rbf, constraints=cons_rbf, method='SLSQP')
alpha_rbf = res_rbf.x

# Support vectors
sv_rbf_idx = np.where((alpha_rbf > 1e-4) & (alpha_rbf < best_C - 1e-4))[0]
if len(sv_rbf_idx) == 0:
    sv_rbf_idx = np.where(alpha_rbf > 1e-4)[0]
b_rbf = np.mean(y_nonlin[sv_rbf_idx] - np.sum(alpha_rbf[:, None] * y_nonlin[:, None] * K_mat[:, sv_rbf_idx], axis=0))

# Decision surface grid
gx = np.linspace(-2.5, 2.5, 150)
gy = np.linspace(-2.5, 2.5, 150)
GX, GY = np.meshgrid(gx, gy)
G_pts = np.column_stack([GX.ravel(), GY.ravel()])

grid_sq_dists = np.sum(G_pts**2, axis=1, keepdims=True) + np.sum(X_nonlin**2, axis=1) - 2 * (G_pts @ X_nonlin.T)
K_grid = np.exp(-best_gamma * np.maximum(0, grid_sq_dists))
Z_rbf = (K_grid @ (alpha_rbf * y_nonlin) + b_rbf).reshape(GX.shape)

ax2.scatter(X_inner[:, 0], X_inner[:, 1], color="royalblue", s=40, label="Class +1 (Inner Ring)")
ax2.scatter(X_outer[:, 0], X_outer[:, 1], color="crimson", s=40, label="Class -1 (Outer Ring)")
ax2.contour(GX, GY, Z_rbf, levels=[-1.0, 0.0, 1.0], colors=["crimson", "black", "royalblue"], 
            linestyles=["--", "-", "--"], linewidths=[1.5, 2.5, 1.5])
ax2.scatter(X_nonlin[sv_rbf_idx, 0], X_nonlin[sv_rbf_idx, 1], s=120, facecolors="none", edgecolors="darkorange", 
            lw=2, label="Support Vectors")

ax2.set_title(rf"Optimal RBF Decision Boundary ($C={best_C}, \gamma={best_gamma}$)", fontsize=12, fontweight="bold")
ax2.set_xlabel("$x_1$")
ax2.set_ylabel("$x_2$")
ax2.legend(loc="upper right", fontsize=9)
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


---
## Exercise 12.4 — Mercer Kernels: Explicit Feature Maps vs. The Kernel Trick

### Problem Statement
In Section 12.4 of *Mathematics for Machine Learning*, a kernel $k: \mathcal{X} \times \mathcal{X} \to \mathbb{R}$ evaluates the inner product between representations of two points in an implicit Reproducing Kernel Hilbert Space (RKHS) $\mathcal{H}$:
$$k(\boldsymbol{x}, \boldsymbol{z}) = \langle \boldsymbol{\phi}(\boldsymbol{x}), \boldsymbol{\phi}(\boldsymbol{z}) \rangle_{\mathcal{H}}$$
By **Mercer's Theorem**, a continuous symmetric function $k(\boldsymbol{x}, \boldsymbol{z})$ is a valid kernel if and only if its Gram matrix $\boldsymbol{K} \in \mathbb{R}^{N \times N}$ with $K_{ij} = k(\boldsymbol{x}_i, \boldsymbol{x}_j)$ is symmetric positive semi-definite (all eigenvalues $\lambda_i \ge 0$) for every finite subset of data points.

In this exercise:
1. **Explicit Feature Map for Quadratic Polynomial Kernel:**
   For 2D inputs $\boldsymbol{x} = [x_1, x_2]^\top, \boldsymbol{z} = [z_1, z_2]^\top$, expand the inhomogeneous quadratic polynomial kernel:
   $$k(\boldsymbol{x}, \boldsymbol{z}) = (\boldsymbol{x}^\top \boldsymbol{z} + c)^2, \quad c > 0$$
   Derive the explicit 6-dimensional non-linear feature map $\boldsymbol{\phi}(\boldsymbol{x}) \in \mathbb{R}^6$ such that $\boldsymbol{\phi}(\boldsymbol{x})^\top \boldsymbol{\phi}(\boldsymbol{z}) \equiv k(\boldsymbol{x}, \boldsymbol{z})$. Prove this identity algebraically and verify it symbolically using SymPy.
2. **Computational Complexity Benchmark:**
   For degree $d$ polynomial kernels in $\mathbb{R}^D$, the explicit feature map has dimension $\binom{D + d}{d}$, which scales as $\mathcal{O}(D^d)$. In contrast, the kernel trick evaluates $k(\boldsymbol{x}, \boldsymbol{z}) = (\boldsymbol{x}^\top \boldsymbol{z} + c)^d$ in $\mathcal{O}(D)$ operations. Benchmark and compare the execution time of computing the Gram matrix via explicit feature coordinates versus the kernel trick as dimension $D$ increases.
3. **Mercer's Positive Semi-Definite Condition:**
   Compute the Gram matrix $\boldsymbol{K}$ on sample data and compute its full eigendecomposition. Verify that all eigenvalues satisfy $\lambda_i \ge 0$.
4. **Infinite-Dimensional Feature Space of the Gaussian RBF:**
   Using the Taylor expansion of the exponential function, prove analytically that the Gaussian RBF kernel:
   $$k(\boldsymbol{x}, \boldsymbol{z}) = \exp\left(-\frac{\|\boldsymbol{x} - \boldsymbol{z}\|^2}{2\sigma^2}\right)$$
   corresponds to an infinite-dimensional feature space $\mathcal{H} = \ell_2$.

---
### Mathematical Derivation

#### 1. Explicit Expansion of Quadratic Polynomial Kernel
Expanding $k(\boldsymbol{x}, \boldsymbol{z}) = (x_1 z_1 + x_2 z_2 + c)^2$:
$$\begin{aligned}
k(\boldsymbol{x}, \boldsymbol{z}) &= (x_1 z_1 + x_2 z_2 + c)(x_1 z_1 + x_2 z_2 + c) \\
&= x_1^2 z_1^2 + x_2^2 z_2^2 + c^2 + 2 x_1 x_2 z_1 z_2 + 2 c x_1 z_1 + 2 c x_2 z_2 \\
&= (x_1^2)(z_1^2) + (x_2^2)(z_2^2) + (\sqrt{2} x_1 x_2)(\sqrt{2} z_1 z_2) + (\sqrt{2c} x_1)(\sqrt{2c} z_1) + (\sqrt{2c} x_2)(\sqrt{2c} z_2) + (c)(c)
\end{aligned}$$
Defining the feature mapping $\boldsymbol{\phi}: \mathbb{R}^2 \to \mathbb{R}^6$:
$$\boldsymbol{\phi}(\boldsymbol{x}) = \begin{bmatrix} x_1^2 \\ x_2^2 \\ \sqrt{2} x_1 x_2 \\ \sqrt{2c} x_1 \\ \sqrt{2c} x_2 \\ c \end{bmatrix} \in \mathbb{R}^6$$
The inner product in feature space is:
$$\boldsymbol{\phi}(\boldsymbol{x})^\top \boldsymbol{\phi}(\boldsymbol{z}) = x_1^2 z_1^2 + x_2^2 z_2^2 + 2 x_1 x_2 z_1 z_2 + 2c x_1 z_1 + 2c x_2 z_2 + c^2 = (\boldsymbol{x}^\top \boldsymbol{z} + c)^2 = k(\boldsymbol{x}, \boldsymbol{z})$$
This proves that linear algorithms in $\mathbb{R}^6$ correspond exactly to quadratic boundaries in $\mathbb{R}^2$.

#### 2. Infinite-Dimensional Feature Space of the Gaussian RBF Kernel
Let $\gamma = \frac{1}{2\sigma^2}$. Using $\|\boldsymbol{x} - \boldsymbol{z}\|^2 = \|\boldsymbol{x}\|^2 + \|\boldsymbol{z}\|^2 - 2 \boldsymbol{x}^\top \boldsymbol{z}$:
$$k(\boldsymbol{x}, \boldsymbol{z}) = \exp(-\gamma \|\boldsymbol{x}\|^2) \exp(-\gamma \|\boldsymbol{z}\|^2) \exp(2\gamma \boldsymbol{x}^\top \boldsymbol{z})$$
Expanding the exponential term via its Taylor series $\exp(u) = \sum_{m=0}^\infty \frac{u^m}{m!}$:
$$\exp(2\gamma \boldsymbol{x}^\top \boldsymbol{z}) = \sum_{m=0}^\infty \frac{(2\gamma)^m}{m!} (\boldsymbol{x}^\top \boldsymbol{z})^m$$
Each term $(\boldsymbol{x}^\top \boldsymbol{z})^m$ is a degree-$m$ homogeneous polynomial kernel, which corresponds to an inner product in a feature space of dimension $\binom{D + m - 1}{m}$. Summing over all integers $m \in \{0, 1, 2, \dots, \infty\}$ produces an infinite sequence of non-linear feature dimensions:
$$\operatorname{dim}(\mathcal{H}_{\text{RBF}}) = \sum_{m=0}^\infty \binom{D + m - 1}{m} = \infty$$
Computing $\boldsymbol{\phi}(\boldsymbol{x})$ explicitly is physically impossible, yet the kernel trick computes their inner product in $\mathcal{O}(D)$ time.


In [ ]:
# 1. SymPy Symbolic Verification of Explicit Polynomial Feature Map
x1, x2, z1, z2, c = symbols('x1 x2 z1 z2 c', real=True)

# Direct kernel expression
k_direct = (x1 * z1 + x2 * z2 + c)**2

# Explicit feature vectors in R^6
phi_x = Matrix([x1**2, x2**2, sqrt(2)*x1*x2, sqrt(2*c)*x1, sqrt(2*c)*x2, c])
phi_z = Matrix([z1**2, z2**2, sqrt(2)*z1*z2, sqrt(2*c)*z1, sqrt(2*c)*z2, c])

# Feature space inner product
k_feature = (phi_x.T * phi_z)[0, 0]

# Symbolic difference check
diff_poly = simplify(k_direct.expand() - k_feature.expand())
print("--- SymPy Symbolic Proof of Mercer Feature Map Equivalence ---")
print(f"Direct Kernel Expansion : {k_direct.expand()}")
print(f"Feature Space Dot Product: {k_feature.expand()}")
print(f"Algebraic Difference     : {diff_poly} (Identically zero: {diff_poly == 0})")

# 2. Mercer Positive Semi-Definiteness Verification
np.random.seed(42)
N_test = 100
X_test = np.random.randn(N_test, 5)

# Compute Gram matrices for Polynomial and RBF kernels
# Polynomial (c=1, d=2)
K_poly = (X_test @ X_test.T + 1.0)**2
eigvals_poly = np.linalg.eigvalsh(K_poly)

# RBF (gamma=0.5)
sq_dists_test = np.sum(X_test**2, axis=1, keepdims=True) + np.sum(X_test**2, axis=1) - 2 * (X_test @ X_test.T)
K_rbf = np.exp(-0.5 * np.maximum(0, sq_dists_test))
eigvals_rbf = np.linalg.eigvalsh(K_rbf)

print("\n--- Mercer Condition: Gram Matrix Eigendecomposition ---")
print(f"Polynomial Kernel: Min Eigenvalue = {np.min(eigvals_poly):.6e} (All >= 0: {np.all(eigvals_poly >= -1e-10)})")
print(f"RBF Kernel       : Min Eigenvalue = {np.min(eigvals_rbf):.6e} (All >= 0: {np.all(eigvals_rbf >= -1e-10)})")


In [ ]:
# 3. Computational Benchmark: Kernel Trick vs Explicit Feature Mapping
def explicit_poly_features_2d(X, c=1.0):
    '''Maps (N, 2) to (N, 6) explicit features.'''
    x1, x2 = X[:, 0], X[:, 1]
    return np.column_stack([
        x1**2, x2**2, np.sqrt(2) * x1 * x2, np.sqrt(2 * c) * x1, np.sqrt(2 * c) * x2, np.full_like(x1, c)
    ])

N_bench_list = [200, 500, 1000, 2000, 4000]
t_kernel_list = []
t_explicit_list = []

for n in N_bench_list:
    X_b = np.random.randn(n, 2)
    
    # Time Kernel Trick: (X @ X.T + 1)^2
    t0 = time.perf_counter()
    K_trick = (X_b @ X_b.T + 1.0)**2
    t_kernel = time.perf_counter() - t0
    t_kernel_list.append(t_kernel)
    
    # Time Explicit Feature Map + Dot Product: Phi(X) @ Phi(X).T
    t0 = time.perf_counter()
    Phi = explicit_poly_features_2d(X_b, c=1.0)
    K_explicit = Phi @ Phi.T
    t_explicit = time.perf_counter() - t0
    t_explicit_list.append(t_explicit)

# Plot benchmark comparison
plt.figure(figsize=(10, 6))
plt.plot(N_bench_list, t_kernel_list, "o-", color="royalblue", lw=2, label=r"Kernel Trick $(\boldsymbol{x}_i^\top \boldsymbol{x}_j + 1)^2$ [$\mathcal{O}(N^2 D)$]")
plt.plot(N_bench_list, t_explicit_list, "s--", color="crimson", lw=2, label=r"Explicit Mapping $\boldsymbol{\Phi} \boldsymbol{\Phi}^\top$ [$\mathcal{O}(N D_{\text{feat}} + N^2 D_{\text{feat}})$]")

plt.title("Computational Scaling: Kernel Trick vs Explicit Feature Mapping", fontsize=12, fontweight="bold")
plt.xlabel("Number of Data Points $N$")
plt.ylabel("Execution Time (seconds)")
plt.legend(loc="upper left")
plt.grid(True, alpha=0.3)
plt.show()

print("Benchmark Results across N points:")
for n, tk, te in zip(N_bench_list, t_kernel_list, t_explicit_list):
    print(f"N = {n:4d} | Kernel Trick: {tk*1000:7.2f} ms | Explicit Mapping: {te*1000:7.2f} ms | Ratio: {te/tk:.2f}x")


---
### Key Takeaways from Chapter 12
1. **Geometric Margin Maximization (Hard-Margin):** Support Vector Machines formulate classification as maximizing the geometric margin $\gamma = \frac{1}{\|\boldsymbol{w}\|}$ between opposing classes. In the dual QP formulation, only data points lying directly on or violating the margin boundaries have non-zero Lagrange multipliers $\alpha_n > 0$ (the support vectors), ensuring sparse solutions.
2. **Soft-Margin Box Constraints & Hinge Loss:** Slack variables $\xi_n \ge 0$ relax linear separability, equivalently expressing the problem as unconstrained Empirical Risk Minimization with Hinge loss. In the dual, this introduces box constraints $0 \le \alpha_n \le C$. Subgradient descent directly minimizes this unconstrained objective and converges to the dual QP decision boundary.
3. **The Kernel Trick & Mercer's Theorem:** Any positive semi-definite Mercer kernel function $k(\boldsymbol{x}_i, \boldsymbol{x}_j)$ computes an inner product in an implicit Hilbert feature space $\mathcal{H}$. This eliminates the need to explicitly compute coordinates in high- or infinite-dimensional spaces (such as the Gaussian RBF), achieving non-linear classification with high computational efficiency.
